# Resources
* https://huggingface.co/learn/nlp-course/chapter1/3?fw=pt
* https://huggingface.co/blog/gemma-peft

# Installing the necessary libraries

In [ ]:
!pip install -q datasets 

# login with your writing token

* you can generate your token here https://huggingface.co/settings/tokens

**IMPORTANT ⚠️** it is recommended you generate a token with writing access so you can save your model into huggingface later

In [ ]:
!pip uninstall -y transformers accelerate huggingface_hub peft trl
!pip install -q \
  transformers==4.41.2 \
  accelerate==0.30.1 \
  huggingface_hub==0.23.5 \
  peft==0.10.0 \
  trl==0.8.6 


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import os

file_path = "/usr/local/lib/python3.11/dist-packages/transformers/models/llama/configuration_llama.py"

if os.path.exists(file_path):
    print("File exists!")
else:
    print("File not found.")


In [ ]:
!pip install --upgrade transformers accelerate peft 

In [ ]:
!pip install --upgrade huggingface_hub

In [ ]:
!pip install --upgrade \
    transformers==5.0.0 \
    huggingface_hub==1.3.7 \
    peft==0.18.1 \
    accelerate==1.12.0 \
    bitsandbytes


In [ ]:
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
lora_config = LoraConfig(
    r=8,                       # low-rank for small fine-tuning
    lora_alpha=16,              # scaling factor
    target_modules=["q_proj", "v_proj"],  # attention modules
    lora_dropout=0.05,          # small dropout
    bias="none",
    task_type="CAUSAL_LM"
)


tokenizer.model_max_length = 2048
model.gradient_checkpointing_enable()
model.enable_input_require_grads()


In [ ]:
for name, param in model.named_parameters():
    # Only LoRA parameters should require grad
    if "lora_" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False


In [ ]:


from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("corbt/enron-emails")

In [ ]:
ds

In [ ]:
df = ds["train"].to_csv

In [ ]:
ds["train"][0]

In [ ]:
import torch
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
messages = [
    {"role": "system", "content": "You are an expert in generating professional enterprise emails "},
    {"role": "user", "content": "Meeting at 7pm in the clubs local room ( recepients  : John Doe , Taher Ben Afia ; sender : oussema ben ameur ) "},
]
outputs = pipe(
    messages,
    max_new_tokens=256,
)
print(outputs[0]["generated_text"][-1])

In [ ]:
eos = tokenizer.eos_token
eos

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
eos = tokenizer.eos_token or "</s>"

def formatting_func(example):
    # Flatten 'to' field and convert all items to str
    to_field = []
    for item in example["to"]:
        if isinstance(item, list):
            to_field.extend(map(str, item))
        else:
            to_field.append(str(item))

    # Flatten 'cc' and 'bcc' fields if needed
    cc_field = [str(x) for x in example.get("cc", []) if x]
    bcc_field = [str(x) for x in example.get("bcc", []) if x]

    # Convert the email to chat-style messages
    messages = [
        {
            "role": "system",
            "content": "You are a helpful AI assistant trained to summarize emails and generate replies."
        },
        {
            "role": "user",
            "content": (
                f"Metadata:\n"
                f"- From: {example['from']}\n"
                f"- To: {', '.join(to_field)}\n"
                f"- Cc: {', '.join(cc_field)}\n"
                f"- Bcc: {', '.join(bcc_field)}\n"
                f"- Subject: {example['subject']}\n\n"
                f"Email:\n{example['body']}\n"
            )
        }
    ]

    # Combine messages into a single string
    text = ""
    for msg in messages:
        text += f"{msg['role'].capitalize()}: {msg['content']}{eos}\n"

    # ✅ Return a list of strings
    return [text]




args = TrainingArguments(
        output_dir="Emails_Generator",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=2,
        max_steps=100,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=1,
        optim="paged_adamw_8bit",
    )

In [ ]:
# Keep only the first 100k examples
train_subset = ds["train"].select(range(100_000))
tokenizer.pad_token = tokenizer.eos_token
# Freeze all base model parameters
for name, param in model.named_parameters():
    if "lora" not in name.lower():
        param.requires_grad = False

# Pass it to SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_subset,
    peft_config=lora_config,
    args=args,
    formatting_func=formatting_func,
    max_seq_length=2048
    
)

trainer.train()


In [ ]:
text = "Subject: Meeting at 6 PM in the Club Local Room"
device = "cuda:0"
inputs = tokenizer(text, return_tensors="pt").to(device)

outputs = trainer.model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0],skip_special_tokens=True))